<a href="https://colab.research.google.com/github/tatasanvi/Experimentation_STT_Bruit_Whisper_Wav2Vec2/blob/main/Experimentation_STT_Bruit_Whisper_Wav2Vec2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch

print("Version de PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU utilisé :", torch.cuda.get_device_name(0))

Version de PyTorch : 2.11.0+cu128
GPU disponible : True
GPU utilisé : Tesla T4


In [10]:
!pip install -q datasets transformers accelerate librosa soundfile jiwer evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.9 MB/s eta 0:00:00


Étape 3 — Charger LibriSpeech

Nous allons maintenant charger 100 échantillons du corpus test-clean. C'est volontairement petit : on vérifie d'abord que toute la chaîne fonctionne avant de passer à une expérimentation plus importante.

In [11]:
from datasets import load_dataset

dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="test[:100]"
)

print(dataset)

README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

clean/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  350MB            

clean/test/0000.parquet: downloading bytes:           |  0.00B            

...

clean/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2620 [00:00<?, ? examples/s]

Generating train.100 split:   0%|          | 0/28539 [00:00<?, ? examples/s]

Generating train.360 split:   0%|          | 0/104014 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [12]:
!df -h
!du -sh /root/.cache/huggingface 2>/dev/null
!du -sh /root/.cache 2>/dev/null

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   76G   37G  68% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G  4.0K  5.7G   1% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1       119G   98G   22G  82% /kaggle/input
tmpfs           6.4G  1.4M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G  7.4G  7.7G  50% /content/drive
29G	/root/.cache/huggingface
29G	/root/.cache


In [13]:
!rm -rf /root/.cache/huggingface

In [14]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G  4.0K  5.7G   1% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1       119G   98G   22G  82% /kaggle/input
tmpfs           6.4G  1.4M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
drive            15G  7.4G  7.7G  50% /content/drive


In [16]:
# ============================================================
# 1. INSTALLATION
# ============================================================

!pip -q install -U datasets soundfile pandas tqdm

2. Nettoyage et vérification du stockage

In [17]:
# ============================================================
# 2. NETTOYAGE DU STOCKAGE
# ============================================================

import os
import shutil
import subprocess

print("========== STOCKAGE AVANT NETTOYAGE ==========")
os.system("df -h /")

# Cache Hugging Face
HF_CACHE = "/root/.cache/huggingface"

if os.path.exists(HF_CACHE):
    print("\nSuppression du cache Hugging Face...")
    shutil.rmtree(HF_CACHE, ignore_errors=True)
    print("Cache Hugging Face supprimé.")
else:
    print("\nAucun cache Hugging Face trouvé.")

# Quelques caches qui peuvent également prendre beaucoup de place
for path in [
    "/root/.cache/pip",
    "/root/.cache/torch",
]:
    if os.path.exists(path):
        print(f"Nettoyage : {path}")
        shutil.rmtree(path, ignore_errors=True)

print("\n========== STOCKAGE APRÈS NETTOYAGE ==========")
os.system("df -h /")

========== STOCKAGE AVANT NETTOYAGE ==========

Aucun cache Hugging Face trouvé.
Nettoyage : /root/.cache/pip

========== STOCKAGE APRÈS NETTOYAGE ==========


0

In [18]:
# ============================================================
# 3. IDENTIFIER LES GROS FICHIERS
# ============================================================

print("========== ESPACE /CONTENT ==========")
os.system("du -h --max-depth=2 /content 2>/dev/null | sort -hr | head -30")

print("\n========== ESPACE /ROOT/.CACHE ==========")
os.system("du -h --max-depth=2 /root/.cache 2>/dev/null | sort -hr | head -30")

========== ESPACE /CONTENT ==========

========== ESPACE /ROOT/.CACHE ==========


0

4. Créer le dossier de travail

In [19]:
# ============================================================
# 4. DOSSIER DE TRAVAIL
# ============================================================

import os

BASE_DIR = "/content/experimentation_transcription"

AUDIO_DIR = os.path.join(BASE_DIR, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)

METADATA_FILE = os.path.join(BASE_DIR, "metadata.csv")

print("Dossier de travail :", BASE_DIR)
print("Dossier audio      :", AUDIO_DIR)

Dossier de travail : /content/experimentation_transcription
Dossier audio      : /content/experimentation_transcription/audio


5. Charger LibriSpeech en STREAMING

In [20]:
# ============================================================
# 5. CHARGEMENT LIBRISPEECH EN STREAMING
# ============================================================

from datasets import load_dataset

dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="test",
    streaming=True
)

print(dataset)

README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

IterableDataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_shards: 1
})


6. Sélectionner exactement 100 audios

In [21]:
# ============================================================
# 6. EXTRACTION DE 100 AUDIOS
# ============================================================

import soundfile as sf
import pandas as pd
from tqdm.auto import tqdm

N_SAMPLES = 100

metadata = []

print(f"Préparation de {N_SAMPLES} audios...")

for i, example in enumerate(tqdm(dataset.take(N_SAMPLES), total=N_SAMPLES)):

    audio = example["audio"]

    # Informations audio
    array = audio["array"]
    sampling_rate = audio["sampling_rate"]

    # Nom du fichier
    filename = f"audio_{i+1:03d}.wav"
    filepath = os.path.join(AUDIO_DIR, filename)

    # Sauvegarde du WAV
    sf.write(
        filepath,
        array,
        sampling_rate
    )

    # Texte de référence
    text = example["text"]

    # Métadonnées
    metadata.append({
        "id": i + 1,
        "filename": filename,
        "text": text,
        "sampling_rate": sampling_rate,
        "duration_seconds": len(array) / sampling_rate
    })

print("\nExtraction terminée.")

Préparation de 100 audios...


  0%|          | 0/100 [00:00<?, ?it/s]


Extraction terminée.


7. Créer le fichier metadata.csv

In [22]:
# ============================================================
# 7. CRÉATION DES MÉTADONNÉES
# ============================================================

df = pd.DataFrame(metadata)

df.to_csv(
    METADATA_FILE,
    index=False,
    encoding="utf-8"
)

print(df.head())
print("\nNombre d'audios :", len(df))
print("Metadata :", METADATA_FILE)

   id       filename                                               text  \
0   1  audio_001.wav     CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS   
1   2  audio_002.wav  THE ENGLISH FORWARDED TO THE FRENCH BASKETS OF...   
2   3  audio_003.wav  CONGRATULATIONS WERE POURED IN UPON THE PRINCE...   
3   4  audio_004.wav  FROM THE RESPECT PAID HER ON ALL SIDES SHE SEE...   
4   5  audio_005.wav  SHE TAUGHT HER DAUGHTER THEN BY HER OWN AFFECT...   

   sampling_rate  duration_seconds  
0          16000             3.505  
1          16000            14.225  
2          16000             5.025  
3          16000            23.315  
4          16000            11.065  

Nombre d'audios : 100
Metadata : /content/experimentation_transcription/metadata.csv


8. Vérifier que nous avons bien 100 audios

In [23]:
audio_files = [
    f for f in os.listdir(AUDIO_DIR)
    if f.lower().endswith(".wav")
]

print("Nombre de fichiers WAV :", len(audio_files))
print("Nombre de lignes CSV   :", len(df))

assert len(audio_files) == 100, "Erreur : le nombre d'audios n'est pas 100."
assert len(df) == 100, "Erreur : le metadata.csv ne contient pas 100 lignes."

print("Les 100 audios sont prêts.")

Nombre de fichiers WAV : 100
Nombre de lignes CSV   : 100
Les 100 audios sont prêts.


9. Vérifier la taille occupée

In [24]:
total_size = 0

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        filepath = os.path.join(root, file)
        total_size += os.path.getsize(filepath)

total_mb = total_size / (1024 ** 2)

print(f"Taille totale du jeu de données : {total_mb:.2f} MB")
print(f"Nombre d'audios : {len(audio_files)}")

Taille totale du jeu de données : 20.48 MB
Nombre d'audios : 100


10. Écouter un exemple
`Pour vérifier que les fichiers sont réellement exploitables :`

In [25]:
# ============================================================
# 10. ÉCOUTER UN AUDIO
# ============================================================

from IPython.display import Audio, display

sample_audio = os.path.join(
    AUDIO_DIR,
    "audio_001.wav"
)

display(Audio(sample_audio))

print("Transcription de référence :")
print(df.iloc[0]["text"])

Transcription de référence :
CONCORD RETURNED TO ITS PLACE AMIDST THE TENTS


11. Afficher les statistiques du jeu

In [26]:
# ============================================================
# 11. STATISTIQUES
# ============================================================

print("========== STATISTIQUES ==========")

print(f"Nombre d'audios       : {len(df)}")
print(f"Durée totale          : {df['duration_seconds'].sum()/60:.2f} minutes")
print(f"Durée moyenne         : {df['duration_seconds'].mean():.2f} secondes")
print(f"Durée minimale        : {df['duration_seconds'].min():.2f} secondes")
print(f"Durée maximale        : {df['duration_seconds'].max():.2f} secondes")
print(f"Fréquence échantill.  : {df['sampling_rate'].unique()}")

========== STATISTIQUES ==========
Nombre d'audios       : 100
Durée totale          : 11.18 minutes
Durée moyenne         : 6.71 secondes
Durée minimale        : 1.81 secondes
Durée maximale        : 23.32 secondes
Fréquence échantill.  : [16000]


Augmentation du corpus pour une experimentation un peu grandeur nature

In [ ]:
streaming=True

In [27]:
# ==============================
# 1. TÉLÉCHARGEMENT VYSTADIAL
# ==============================

import os
import shutil
import tarfile
import urllib.request

BASE = "/content/thesis_data"
VYSTADIAL_DIR = os.path.join(BASE, "vystadial")

os.makedirs(VYSTADIAL_DIR, exist_ok=True)

url = "https://www.openslr.org/resources/6/data_voip_en.tgz"
archive = os.path.join(BASE, "vystadial_en.tgz")

print("Téléchargement...")
urllib.request.urlretrieve(url, archive)

print("Extraction...")
with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(VYSTADIAL_DIR)

print("Nettoyage de l'archive...")
os.remove(archive)

print("Terminé.")
print("Taille du dossier :")

total = 0
for root, dirs, files in os.walk(VYSTADIAL_DIR):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))

print(f"{total / (1024**3):.2f} Go")

Téléchargement...
Extraction...


/tmp/ipykernel_2075/2367658950.py:23: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(VYSTADIAL_DIR)


Nettoyage de l'archive...
Terminé.
Taille du dossier :
9.67 Go


In [28]:
import os

for root, dirs, files in os.walk("/content/thesis_data/vystadial"):
    level = root.replace("/content/thesis_data/vystadial", "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}  {file}")

    if level >= 2:
        dirs[:] = []

vystadial/
  data_voip_en/
    LICENSE.TXT
    README.rst
    arpa_bigram
    .history-oplatek
    data/
      jurcic-006-130711_182925_0006502_0006661.wav
      jurcic-003-130706_170511_0005132_0005255.wav.trn
      jurcic-001-130716_173453_0001935_0002344.wav
      jurcic-004-121019_115545_0003893_0004005.wav.trn
      jurcic-005-130702_023817_0006036_0006339.wav
      jurcic-003-130713_032827_0003851_0004137.wav
      jurcic-002-130712_195625_0003574_0003657.wav
      jurcic-011-130721_013716_0006729_0006860.wav.trn
      jurcic-004-130708_153253_0002291_0002450.wav.trn
      jurcic-009-121025_092943_0010524_0010663.wav.trn
    dev/
      jurcic-003-130720_204732_0000946_0001275.wav.trn
      jurcic-015-130726_190002_0014674_0014841.wav
      jurcic-006-130722_163458_0008489_0008828.wav
      jurcic-004-130711_180535_0007988_0008317.wav
      jurcic-001-130723_170809_0001862_0002272.wav.trn
      jurcic-003-130715_091119_0002765_0003059.wav.trn
      jurcic-004-130707_184544_0003035

Audit complet de Vystadial

In [29]:
import os
import wave
import pandas as pd
from tqdm.auto import tqdm

BASE = "/content/thesis_data/vystadial/data_voip_en"

splits = ["train", "dev", "test"]

records = []

for split in splits:
    split_dir = os.path.join(BASE, split)

    wav_files = [
        f for f in os.listdir(split_dir)
        if f.lower().endswith(".wav")
    ]

    print(f"\n{split.upper()} : {len(wav_files)} fichiers WAV")

    for filename in tqdm(wav_files, desc=split):
        wav_path = os.path.join(split_dir, filename)
        trn_path = wav_path + ".trn"

        if not os.path.exists(trn_path):
            records.append({
                "split": split,
                "audio": filename,
                "transcript": None,
                "sample_rate": None,
                "duration": None,
                "channels": None,
                "sample_width": None,
                "has_transcript": False
            })
            continue

        # Lecture des caractéristiques audio
        try:
            with wave.open(wav_path, "rb") as wf:
                sample_rate = wf.getframerate()
                frames = wf.getnframes()
                duration = frames / sample_rate
                channels = wf.getnchannels()
                sample_width = wf.getsampwidth()
        except Exception:
            sample_rate = None
            duration = None
            channels = None
            sample_width = None

        # Lecture transcription
        try:
            with open(trn_path, "r", encoding="utf-8", errors="ignore") as f:
                transcript = f.read().strip()
        except Exception:
            transcript = None

        records.append({
            "split": split,
            "audio": filename,
            "transcript": transcript,
            "sample_rate": sample_rate,
            "duration": duration,
            "channels": channels,
            "sample_width": sample_width,
            "has_transcript": bool(transcript)
        })

df = pd.DataFrame(records)

print("\n==============================")
print("AUDIT VYSTADIAL")
print("==============================")

print("Nombre total WAV :", len(df))
print("Avec transcription :", df["has_transcript"].sum())
print("Sans transcription :", (~df["has_transcript"]).sum())

print("\nFréquences d'échantillonnage :")
print(df["sample_rate"].value_counts(dropna=False))

print("\nNombre de canaux :")
print(df["channels"].value_counts(dropna=False))

print("\nDurée totale :")
print(round(df["duration"].sum() / 3600, 2), "heures")

print("\nDurée moyenne :")
print(round(df["duration"].mean(), 2), "secondes")

print("\nDurée min/max :")
print(
    round(df["duration"].min(), 2),
    "→",
    round(df["duration"].max(), 2),
    "secondes"
)

print("\nExemples de transcriptions :")
display(df[df["has_transcript"]][
    ["split", "audio", "transcript", "duration"]
].head(10))


TRAIN : 47463 fichiers WAV


train:   0%|          | 0/47463 [00:00<?, ?it/s]


DEV : 2000 fichiers WAV


dev:   0%|          | 0/2000 [00:00<?, ?it/s]


TEST : 2000 fichiers WAV


test:   0%|          | 0/2000 [00:00<?, ?it/s]


AUDIT VYSTADIAL
Nombre total WAV : 51463
Avec transcription : 51463
Sans transcription : 0

Fréquences d'échantillonnage :
sample_rate
16000    51463
Name: count, dtype: int64

Nombre de canaux :
channels
1    51463
Name: count, dtype: int64

Durée totale :
45.02 heures

Durée moyenne :
3.15 secondes

Durée min/max :
0.45 → 90.43 secondes

Exemples de transcriptions :


,split,audio,transcript,duration
0,train,jurcic-006-130711_182925_0006502_0006661.wav,AND WHAT IS THE POST CODE,2.624
1,train,jurcic-001-130716_173453_0001935_0002344.wav,I'M LOOKING FOR A MODERATELY PRICED RESTAURANT...,5.184
2,train,jurcic-005-130702_023817_0006036_0006339.wav,CAN I HAVE THE ADDRESS OF THE VENUE,4.096
3,train,jurcic-003-130713_032827_0003851_0004137.wav,GIVE ME THE ADDRESS PHONE NUMBER AND PRICES,3.904
4,train,jurcic-002-130712_195625_0003574_0003657.wav,NO,1.920
5,train,jurcic-004-130707_160229_0004410_0004543.wav,IT DOES NOT MATTER,2.368
6,train,jurcic-004-120925_040217_0001845_0001930.wav,A PUB,1.920
7,train,jurcic-002-130711_185924_0003687_0003786.wav,WHAT IS THE ADDRESS,2.048
8,train,jurcic-001-130711_185715_0002045_0002408.wav,I NEED A PUB WITH AN INTERNET CONNECTION AND A...,4.736
9,train,jurcic-004-120925_222828_0005847_0006078.wav,WHAT IS THE POST CODE,3.392


Sauvegarder uniquement les métadonnées

In [30]:
metadata_path = "/content/thesis_data/vystadial_metadata.csv"

df.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8"
)

print(f"Metadata sauvegardées : {metadata_path}")
print(f"Taille : {os.path.getsize(metadata_path)/1024:.1f} Ko")

Metadata sauvegardées : /content/thesis_data/vystadial_metadata.csv
Taille : 4913.1 Ko


In [31]:
# ==========================================
# ÉTAPE 5 — ANALYSE DÉTAILLÉE VYSTADIAL
# ==========================================

import pandas as pd
import os
import re

metadata_path = "/content/thesis_data/vystadial_metadata.csv"

df = pd.read_csv(metadata_path)

# ------------------------------------------
# 1. EXTRAIRE L'IDENTIFIANT DU LOCUTEUR
# ------------------------------------------

df["speaker"] = df["audio"].str.extract(
    r"^([^-]+-\d+)"
)

# ------------------------------------------
# 2. STATISTIQUES PAR SPLIT
# ------------------------------------------

print("===== RÉPARTITION =====")

print(
    df.groupby("split").agg(
        segments=("audio", "count"),
        heures=("duration", lambda x: x.sum() / 3600),
        duree_moyenne=("duration", "mean")
    )
)

# ------------------------------------------
# 3. LOCUTEURS
# ------------------------------------------

print("\n===== LOCUTEURS =====")

print("Nombre de locuteurs :", df["speaker"].nunique())

print("\nSegments par locuteur :")

speaker_stats = (
    df.groupby("speaker")
      .agg(
          segments=("audio", "count"),
          duree=("duration", "sum")
      )
      .sort_values("segments", ascending=False)
)

display(speaker_stats.head(20))

# ------------------------------------------
# 4. DURÉE
# ------------------------------------------

print("\n===== DURÉE =====")

print("Minimum :", round(df["duration"].min(), 2), "s")
print("Moyenne :", round(df["duration"].mean(), 2), "s")
print("Médiane :", round(df["duration"].median(), 2), "s")
print("Maximum :", round(df["duration"].max(), 2), "s")

print("\nPercentiles :")

print(
    df["duration"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# ------------------------------------------
# 5. LONGUEUR DES TRANSCRIPTIONS
# ------------------------------------------

df["transcript"] = df["transcript"].fillna("").astype(str)

df["words"] = df["transcript"].apply(
    lambda x: len(x.split())
)

df["characters"] = df["transcript"].apply(
    len
)

print("\n===== TRANSCRIPTIONS =====")

print("Mots moyen :", round(df["words"].mean(), 2))
print("Mots médian :", round(df["words"].median(), 2))
print("Minimum mots :", df["words"].min())
print("Maximum mots :", df["words"].max())

# ------------------------------------------
# 6. TRANSCRIPTIONS VIDES / ANORMALES
# ------------------------------------------

empty = df[df["transcript"].str.strip() == ""]

print("\nTranscriptions vides :", len(empty))

# ------------------------------------------
# 7. SEGMENTS COURTS / LONGS
# ------------------------------------------

print("\n===== SEGMENTS =====")

print("Moins de 1 seconde :", (df["duration"] < 1).sum())
print("1–2 secondes :", ((df["duration"] >= 1) & (df["duration"] < 2)).sum())
print("2–5 secondes :", ((df["duration"] >= 2) & (df["duration"] < 5)).sum())
print("5–10 secondes :", ((df["duration"] >= 5) & (df["duration"] < 10)).sum())
print("Plus de 10 secondes :", (df["duration"] >= 10).sum())

# ------------------------------------------
# 8. EXEMPLES DE TRANSCRIPTIONS
# ------------------------------------------

print("\n===== EXEMPLES =====")

display(
    df[
        ["split", "speaker", "audio", "duration", "words", "transcript"]
    ].sample(
        min(20, len(df)),
        random_state=42
    )
)

===== RÉPARTITION =====
       segments     heures  duree_moyenne
split                                    
dev        2000   1.751200       3.152160
test       2000   1.763858       3.174944
train     47463  41.506356       3.148197

===== LOCUTEURS =====
Nombre de locuteurs : 30

Segments par locuteur :


,segments,duree
speaker,,
jurcic-003,8210,23960.320
jurcic-002,8077,25960.896
jurcic-001,7292,32952.064
jurcic-004,7040,20008.832
jurcic-005,5323,15044.608
jurcic-006,3855,10977.792
jurcic-007,2885,8088.384
jurcic-008,2172,6247.040
jurcic-009,1640,4672.256



===== DURÉE =====
Minimum : 0.45 s
Moyenne : 3.15 s
Médiane : 2.62 s
Maximum : 90.43 s

Percentiles :
0.01    1.472
0.05    1.792
0.25    2.304
0.50    2.624
0.75    3.648
0.95    5.696
0.99    7.616
Name: duration, dtype: float64

===== TRANSCRIPTIONS =====
Mots moyen : 4.76
Mots médian : 4.0
Minimum mots : 0
Maximum mots : 40

Transcriptions vides : 1

===== SEGMENTS =====
Moins de 1 seconde : 63
1–2 secondes : 5543
2–5 secondes : 41272
5–10 secondes : 4398
Plus de 10 secondes : 187

===== EXEMPLES =====


,split,speaker,audio,duration,words,transcript
18659,train,jurcic-004,jurcic-004-130701_181436_0003806_0004066.wav,3.648,7,WHAT IS THE ADDRESS AND PHONE NUMBER
8557,train,jurcic-008,jurcic-008-130724_171214_0006139_0006274.wav,2.432,5,WHAT IS THE PHONE NUMBER
21722,train,jurcic-007,jurcic-007-130714_000929_0008538_0008856.wav,4.224,8,I'M LOOKING FOR AN EXPENSIVE RESTAURANT IN CAS...
4144,train,jurcic-015,jurcic-015-130715_164130_0009599_0009744.wav,2.496,3,AND PHONE NUMBER
18533,train,jurcic-026,jurcic-026-120925_184936_0018747_0019047.wav,4.096,2,UGLY DUCKLING
33570,train,jurcic-011,jurcic-011-120925_035938_0014045_0014109.wav,1.728,3,THANK YOU GOODBYE
47350,train,jurcic-003,jurcic-003-120930_225445_0003812_0003884.wav,1.792,3,WHAT'S THE ADDRESS
45333,train,jurcic-001,jurcic-001-120924_214126_0000121_0000402.wav,3.840,1,_NOISE_
22501,train,jurcic-001,jurcic-001-130716_032515_0001952_0002459.wav,6.144,14,I'M LOOKING FOR A MODERATELY PRICED RESTAURANT...
45024,train,jurcic-004,jurcic-004-130704_212312_0003708_0003897.wav,2.944,3,THE FENDITTON AREA


In [32]:
# ==========================================
# ÉTAPE 6 — TÉLÉCHARGEMENT MEDIASPEECH FR
# ==========================================

import os
import requests

base_dir = "/content/thesis_data"
os.makedirs(base_dir, exist_ok=True)

url = "https://www.openslr.org/resources/108/FR.tgz"
output = os.path.join(base_dir, "mediaspeech_fr.tgz")

print("Téléchargement de MediaSpeech FR...")
print("Destination :", output)

r = requests.get(url, stream=True)
r.raise_for_status()

total = int(r.headers.get("content-length", 0))
downloaded = 0

with open(output, "wb") as f:
    for chunk in r.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)
            downloaded += len(chunk)

            if total:
                percent = downloaded / total * 100
                print(
                    f"\rProgression : {percent:.1f}%",
                    end=""
                )

print("\n\nTéléchargement terminé.")
print("Taille :", round(os.path.getsize(output) / (1024**2), 2), "MB")

Téléchargement de MediaSpeech FR...
Destination : /content/thesis_data/mediaspeech_fr.tgz
Progression : 100.0%

Téléchargement terminé.
Taille : 608.16 MB


In [33]:
# ==========================================
# EXTRACTION
# ==========================================

import tarfile
import os

extract_dir = os.path.join(base_dir, "mediaspeech_fr")

os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(output, "r:gz") as tar:
    tar.extractall(extract_dir)

print("Extraction terminée.")
print("Contenu :")

for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    if level >= 2:
        dirs[:] = []

    for file in files[:10]:
        print(f"{indent}  {file}")

/tmp/ipykernel_2075/2049935401.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


Extraction terminée.
Contenu :
mediaspeech_fr/
  FR/
    254ff50f-f9e3-4186-bca2-2c7bc99a4d6c.flac
    06de4e33-9340-4c54-9404-d0b96d40bbf5.flac
    f0a5f120-ac95-4736-ac5c-3523d65d9fd3.flac
    0b8d6d15-616f-423e-bcf1-c15f6e64b060.txt
    ee06348c-998e-4f49-8866-5f33c2f64ee4.txt
    815a980b-ad33-407f-9a00-bcf884e62987.flac
    cb08c981-4e08-4cae-a400-934148dd87d9.flac
    461337da-0054-414a-9bc0-5b8a26b0ac4b.txt
    e0b40742-0ef1-4b0f-b8a3-c725162e36ab.flac
    f47e6bff-c6dd-42e8-85c5-94f0c3f8f605.txt


In [34]:
# ============================================================
# ÉTAPE 7 — AUDIT COMPLET MEDIASPEECH FR
# ============================================================

import os
import glob
import pandas as pd
import soundfile as sf
from collections import Counter

base_dir = "/content/thesis_data/mediaspeech_fr/FR"

print("=" * 60)
print("AUDIT MEDIASPEECH FR")
print("=" * 60)

# ------------------------------------------------------------
# 1. Fichiers
# ------------------------------------------------------------

audio_files = glob.glob(os.path.join(base_dir, "*.flac"))
text_files = glob.glob(os.path.join(base_dir, "*.txt"))

print("\n===== FICHIERS =====")
print("Fichiers audio :", len(audio_files))
print("Fichiers texte :", len(text_files))

# ------------------------------------------------------------
# 2. Correspondance audio / transcription
# ------------------------------------------------------------

audio_ids = {
    os.path.splitext(os.path.basename(f))[0]
    for f in audio_files
}

text_ids = {
    os.path.splitext(os.path.basename(f))[0]
    for f in text_files
}

missing_text = audio_ids - text_ids
orphan_text = text_ids - audio_ids

print("\n===== CORRESPONDANCE =====")
print("Audio avec transcription :", len(audio_ids & text_ids))
print("Audio sans transcription :", len(missing_text))
print("Textes sans audio :", len(orphan_text))

if missing_text:
    print("\nExemples audio sans transcription :")
    print(list(missing_text)[:10])

if orphan_text:
    print("\nExemples textes sans audio :")
    print(list(orphan_text)[:10])

# ------------------------------------------------------------
# 3. Lecture des métadonnées audio
# ------------------------------------------------------------

records = []

print("\n===== ANALYSE AUDIO =====")

for i, audio_path in enumerate(audio_files):

    try:
        info = sf.info(audio_path)

        duration = info.frames / info.samplerate

        records.append({
            "id": os.path.splitext(os.path.basename(audio_path))[0],
            "audio": audio_path,
            "duration": duration,
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "frames": info.frames,
            "format": info.format,
            "subtype": info.subtype
        })

    except Exception as e:
        print("Erreur :", audio_path, e)

    if (i + 1) % 1000 == 0:
        print(f"\rAnalysé : {i+1}/{len(audio_files)}", end="")

print("\nAnalyse terminée.")

df = pd.DataFrame(records)

# ------------------------------------------------------------
# 4. Statistiques audio
# ------------------------------------------------------------

print("\n===== CARACTÉRISTIQUES AUDIO =====")

print("Nombre de fichiers :", len(df))

print("\nFréquences d'échantillonnage :")
print(df["sample_rate"].value_counts())

print("\nNombre de canaux :")
print(df["channels"].value_counts())

print("\nFormats :")
print(df["format"].value_counts())

print("\nSubtypes :")
print(df["subtype"].value_counts())

print("\n===== DURÉE =====")

total_seconds = df["duration"].sum()

print("Durée totale :", round(total_seconds / 3600, 3), "heures")
print("Durée moyenne :", round(df["duration"].mean(), 3), "secondes")
print("Durée médiane :", round(df["duration"].median(), 3), "secondes")
print("Durée minimale :", round(df["duration"].min(), 3), "secondes")
print("Durée maximale :", round(df["duration"].max(), 3), "secondes")

print("\nPercentiles :")
print(
    df["duration"].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# ------------------------------------------------------------
# 5. Transcriptions
# ------------------------------------------------------------

print("\n===== TRANSCRIPTIONS =====")

transcripts = {}

for txt_path in text_files:

    file_id = os.path.splitext(os.path.basename(txt_path))[0]

    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        transcripts[file_id] = text

    except UnicodeDecodeError:

        with open(txt_path, "r", encoding="latin-1") as f:
            text = f.read().strip()

        transcripts[file_id] = text

df["transcript"] = df["id"].map(transcripts)

df["transcript"] = df["transcript"].fillna("").astype(str)

df["words"] = df["transcript"].apply(
    lambda x: len(x.split())
)

df["characters"] = df["transcript"].apply(len)

print("Transcriptions disponibles :", (df["transcript"].str.strip() != "").sum())
print("Transcriptions vides :", (df["transcript"].str.strip() == "").sum())

print("\nMots moyen :", round(df["words"].mean(), 2))
print("Mots médian :", round(df["words"].median(), 2))
print("Minimum mots :", df["words"].min())
print("Maximum mots :", df["words"].max())

# ------------------------------------------------------------
# 6. Distribution des durées
# ------------------------------------------------------------

print("\n===== DISTRIBUTION DES SEGMENTS =====")

print(
    "< 1 seconde :",
    (df["duration"] < 1).sum()
)

print(
    "1–2 secondes :",
    ((df["duration"] >= 1) &
     (df["duration"] < 2)).sum()
)

print(
    "2–5 secondes :",
    ((df["duration"] >= 2) &
     (df["duration"] < 5)).sum()
)

print(
    "5–10 secondes :",
    ((df["duration"] >= 5) &
     (df["duration"] < 10)).sum()
)

print(
    "> 10 secondes :",
    (df["duration"] >= 10).sum()
)

# ------------------------------------------------------------
# 7. Exemples
# ------------------------------------------------------------

print("\n===== EXEMPLES =====")

display(
    df[
        [
            "id",
            "duration",
            "sample_rate",
            "channels",
            "words",
            "transcript"
        ]
    ].sample(
        min(20, len(df)),
        random_state=42
    )
)

# ------------------------------------------------------------
# 8. Sauvegarde
# ------------------------------------------------------------

metadata_path = "/content/thesis_data/mediaspeech_fr_metadata.csv"

df.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8"
)

print("\n===== SAUVEGARDE =====")
print(metadata_path)
print("Taille :", round(os.path.getsize(metadata_path) / 1024**2, 2), "MB")

AUDIT MEDIASPEECH FR

===== FICHIERS =====
Fichiers audio : 2498
Fichiers texte : 2498

===== CORRESPONDANCE =====
Audio avec transcription : 2498
Audio sans transcription : 0
Textes sans audio : 0

===== ANALYSE AUDIO =====
Analysé : 2000/2498
Analyse terminée.

===== CARACTÉRISTIQUES AUDIO =====
Nombre de fichiers : 2498

Fréquences d'échantillonnage :
sample_rate
16000    2498
Name: count, dtype: int64

Nombre de canaux :
channels
1    2498
Name: count, dtype: int64

Formats :
format
FLAC    2498
Name: count, dtype: int64

Subtypes :
subtype
PCM_16    2498
Name: count, dtype: int64

===== DURÉE =====
Durée totale : 10.0 heures
Durée moyenne : 14.412 secondes
Durée médiane : 14.7 secondes
Durée minimale : 4.1 secondes
Durée maximale : 14.9 secondes

Percentiles :
0.01     9.194
0.05    12.800
0.25    14.500
0.50    14.700
0.75    14.900
0.95    14.900
0.99    14.900
Name: duration, dtype: float64

===== TRANSCRIPTIONS =====
Transcriptions disponibles : 2498
Transcriptions vides : 0



,id,duration,sample_rate,channels,words,transcript
2293,1ded9548-62c1-42f9-ac48-6412d2fcd65e,14.5,16000,1,45,le comptable à tout le monde l'utiliser et qu'...
1864,4e7e5b89-1064-4837-8b87-fb39704697fb,14.9,16000,1,59,non pas parce qu'elles étaient les plus fort m...
902,42f00b1a-d432-4c2d-818f-885cf544b186,14.8,16000,1,48,membre de force ouvrière au micro de cynthia l...
2239,6333d368-1869-4e4e-be73-0b8bd3298e0f,14.9,16000,1,44,courant mais mais ce que le président américai...
1285,dcedf589-684e-4dd9-891a-d7155fcb52f5,14.8,16000,1,38,sont intéressants pour cette application c'est...
56,eac90ef7-8550-438b-9d17-af0727f6db21,14.8,16000,1,41,a affirmé ne pas avoir besoin de ce texte même...
1988,ca2cc94e-2906-465e-920e-6755fb94525f,14.1,16000,1,37,blic s'est déchainé on en parlait à l'instant ...
802,4eeac754-b94c-4612-9ee4-dac3848ce635,14.6,16000,1,38,secteur comme les vôtres c'est compréhensible...
812,5ccd1ddd-1c3b-439b-826f-ccdabddb3b27,14.8,16000,1,41,et c'est avant tout une question de souveraine...
903,40988962-7a8a-4932-84cb-2ca695d63da5,14.9,16000,1,48,qualifié celleci malheureusement elle n'existe...



===== SAUVEGARDE =====
/content/thesis_data/mediaspeech_fr_metadata.csv
Taille : 0.95 MB


In [35]:
# ============================================================
# ÉTAPE 8 — TÉLÉCHARGEMENT NICOLINGUA SLR106
# ============================================================

import os
import requests

base_dir = "/content/thesis_data"
os.makedirs(base_dir, exist_ok=True)

url = "https://www.openslr.org/resources/106/nicolingua-0004-west-african-va-asr-corpus.tgz"

output = os.path.join(
    base_dir,
    "nicolingua.tgz"
)

print("Téléchargement Nicolingua SLR106...")
print("Destination :", output)

r = requests.get(url, stream=True)
r.raise_for_status()

total = int(r.headers.get("content-length", 0))
downloaded = 0

with open(output, "wb") as f:
    for chunk in r.iter_content(chunk_size=1024 * 1024):

        if chunk:

            f.write(chunk)
            downloaded += len(chunk)

            if total:
                percent = downloaded / total * 100
                print(
                    f"\rProgression : {percent:.1f}%",
                    end=""
                )

print("\n\nTéléchargement terminé.")

print(
    "Taille :",
    round(os.path.getsize(output) / (1024**2), 2),
    "MB"
)

Téléchargement Nicolingua SLR106...
Destination : /content/thesis_data/nicolingua.tgz
Progression : 100.0%

Téléchargement terminé.
Taille : 243.12 MB


In [36]:
# ============================================================
# EXTRACTION NICOLINGUA
# ============================================================

import tarfile
import os

extract_dir = os.path.join(
    base_dir,
    "nicolingua"
)

os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(output, "r:gz") as tar:
    tar.extractall(extract_dir)

print("Extraction terminée.")

for root, dirs, files in os.walk(extract_dir):

    level = root.replace(extract_dir, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    if level >= 2:
        dirs[:] = []

    for file in files[:10]:
        print(f"{indent}  {file}")

/tmp/ipykernel_2075/1475639740.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(extract_dir)


Extraction terminée.
nicolingua/
  nicolingua-0004-west-african-va-asr-corpus/
    LICENSE.txt
    data/


In [37]:
import os

root = "/content/thesis_data/nicolingua"

for dirpath, dirnames, filenames in os.walk(root):
    level = dirpath.replace(root, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    for f in filenames[:10]:
        print(f"{indent}  {f}")

nicolingua/
  nicolingua-0004-west-african-va-asr-corpus/
    LICENSE.txt
    data/
      audio_samples/
        r010_s007_d002_susu_524_hadja.wav
        r124_s045_d008_susu_207_no.wav
        r109_s040_d004_maninka_402_dad.wav
        r069_s027_d003_francais_514_fanta.wav
        r068_s027_d002_maninka_517_ousmane.wav
        r035_s017_d001_susu_304_three.wav
        r088_s033_d003_susu_304_three.wav
        r086_s032_d002_pular_207_no.wav
        r060_s025_d003_pular_202_search_contact.wav
        r088_s033_d003_susu_513_amadou.wav
      meta/
        metadata.csv
        vocab_names.csv
        devices_a.csv
        vocab_parents.csv
        recording_sessions_a.csv
        speakers_a.csv
        vocab_contact_management.csv
        vocab_digits.csv
        vocab_wake_words.csv


In [38]:
# ============================================================
# ÉTAPE 8 — LOCALISATION AUTOMATIQUE DE NICOLINGUA
# ============================================================

import os

base = "/content/thesis_data/nicolingua"

print("=" * 70)
print("CONTENU RÉEL DE NICOLINGUA")
print("=" * 70)

if not os.path.exists(base):
    print("ERREUR : le dossier n'existe pas :", base)
else:

    for root, dirs, files in os.walk(base):

        level = root.replace(base, "").count(os.sep)
        indent = "  " * level

        print(f"{indent}{os.path.basename(root)}/")

        # Afficher les fichiers présents
        for f in sorted(files)[:20]:
            print(f"{indent}  {f}")

        # Ne pas afficher une arborescence énorme
        if level >= 3:
            dirs[:] = []

CONTENU RÉEL DE NICOLINGUA
nicolingua/
  nicolingua-0004-west-african-va-asr-corpus/
    LICENSE.txt
    data/
      audio_samples/
        r001_s001_d001_maninka_101_wake_word.wav
        r001_s001_d001_maninka_201_add_contact.wav
        r001_s001_d001_maninka_202_search_contact.wav
        r001_s001_d001_maninka_203_update_contact.wav
        r001_s001_d001_maninka_204_delete_contact.wav
        r001_s001_d001_maninka_205_call_contact.wav
        r001_s001_d001_maninka_206_yes.wav
        r001_s001_d001_maninka_207_no.wav
        r001_s001_d001_maninka_301_zero.wav
        r001_s001_d001_maninka_302_one.wav
        r001_s001_d001_maninka_303_two.wav
        r001_s001_d001_maninka_304_three.wav
        r001_s001_d001_maninka_305_four.wav
        r001_s001_d001_maninka_306_five.wav
        r001_s001_d001_maninka_307_six.wav
        r001_s001_d001_maninka_308_seven.wav
        r001_s001_d001_maninka_309_eight.wav
        r001_s001_d001_maninka_310_nine.wav
        r001_s001_d001_manink

In [39]:
# ============================================================
# RECHERCHE DES FICHIERS NICOLINGUA
# ============================================================

import os
from collections import Counter

base = "/content/thesis_data/nicolingua"

extensions = Counter()
files_by_ext = {}

for root, dirs, files in os.walk(base):

    for file in files:

        ext = os.path.splitext(file)[1].lower()

        extensions[ext] += 1

        if ext not in files_by_ext:
            files_by_ext[ext] = []

        files_by_ext[ext].append(
            os.path.join(root, file)
        )

print("=" * 70)
print("EXTENSIONS TROUVÉES")
print("=" * 70)

for ext, count in extensions.most_common():
    print(f"{ext or '[sans extension]':15s} : {count}")

print("\n" + "=" * 70)
print("EXEMPLES PAR TYPE")
print("=" * 70)

for ext, files in files_by_ext.items():

    print(f"\n--- {ext or '[sans extension]'} ---")

    for f in files[:10]:
        print(f)

EXTENSIONS TROUVÉES
.wav            : 10083
.csv            : 9
.txt            : 1

EXEMPLES PAR TYPE

--- .txt ---
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/LICENSE.txt

--- .wav ---
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/data/audio_samples/r010_s007_d002_susu_524_hadja.wav
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/data/audio_samples/r124_s045_d008_susu_207_no.wav
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/data/audio_samples/r109_s040_d004_maninka_402_dad.wav
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/data/audio_samples/r069_s027_d003_francais_514_fanta.wav
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/data/audio_samples/r068_s027_d002_maninka_517_ousmane.wav
/content/thesis_data/nicolingua/nicolingua-0004-west-african-va-asr-corpus/data/audio_samples/r035_s017_d001_susu_304_three.wav
/conten

In [40]:
# ============================================================
# DIAGNOSTIC — EMPLACEMENT RÉEL DE NICOLINGUA
# ============================================================

import os

print("=" * 70)
print("CONTENU DE /content/thesis_data")
print("=" * 70)

base = "/content/thesis_data"

if not os.path.exists(base):
    print("ERREUR : /content/thesis_data n'existe pas")
else:
    for item in sorted(os.listdir(base)):
        path = os.path.join(base, item)

        if os.path.isdir(path):
            print(f"[DOSSIER] {item}")
        else:
            size = os.path.getsize(path) / (1024**2)
            print(f"[FICHIER] {item} — {size:.2f} MB")


print("\n" + "=" * 70)
print("RECHERCHE DE NICOLINGUA")
print("=" * 70)

matches = []

for root, dirs, files in os.walk("/content"):

    for name in files:

        if "nicolingua" in name.lower():
            path = os.path.join(root, name)
            matches.append(path)

        if "nicolingua" in root.lower() and len(matches) < 50:
            pass

print("Fichiers trouvés contenant 'nicolingua' :")

for path in matches[:50]:
    print(path)


print("\n" + "=" * 70)
print("RECHERCHE DES ARCHIVES .TGZ")
print("=" * 70)

for root, dirs, files in os.walk("/content"):

    for name in files:

        if name.lower().endswith((".tgz", ".tar.gz")):
            path = os.path.join(root, name)
            size = os.path.getsize(path) / (1024**2)

            print(
                f"{path} — {size:.2f} MB"
            )

CONTENU DE /content/thesis_data
[DOSSIER] mediaspeech_fr
[FICHIER] mediaspeech_fr.tgz — 608.16 MB
[FICHIER] mediaspeech_fr_metadata.csv — 0.95 MB
[DOSSIER] nicolingua
[FICHIER] nicolingua.tgz — 243.12 MB
[DOSSIER] vystadial
[FICHIER] vystadial_metadata.csv — 4.80 MB

RECHERCHE DE NICOLINGUA
Fichiers trouvés contenant 'nicolingua' :
/content/thesis_data/nicolingua.tgz

RECHERCHE DES ARCHIVES .TGZ
/content/thesis_data/nicolingua.tgz — 243.12 MB
/content/thesis_data/mediaspeech_fr.tgz — 608.16 MB
